In [15]:
import numpy as np
import pandas as pd
from numpy import nan
from sklearn.preprocessing import  StandardScaler, OneHotEncoder
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA, TruncatedSVD
from nltk.stem import SnowballStemmer
from sklearn import linear_model, datasets
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import RandomizedSearchCV

import nltk
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [16]:
df = pd.read_csv("winter_project_2026/development.csv")
eva= pd.read_csv("winter_project_2026/evaluation.csv")

In [17]:
rows = df.loc[df.duplicated(subset=['article', 'title'])]
duplicated = df.loc[df.duplicated(subset=['title','article'])]
mask = df['article'].isin(duplicated['article'])
to_drop = df[mask]
df.drop(to_drop["Id"], inplace=True)
df.dropna(inplace=True)

In [18]:
y = df['label']
df.drop('label', inplace=True, axis=1)

In [19]:
# unisco le colonne titolo e articolo in una colonna unica raddoppiando l'importanza del titolo
text = df['title'] + ' ' + df['title'] + ' ' + df['article']
df = pd.concat([df, text], axis=1)

#rinomino le colonne e droppo title e article
df.rename(columns={0: 'text'}, inplace=True)
df.drop(['title', 'article'], inplace=True, axis=1)

df.columns

Index(['Id', 'source', 'page_rank', 'timestamp', 'text'], dtype='object')

In [20]:
X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y, shuffle=True)

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np


TOP_K_PER_CLASS = 7000 
final_vocabulary = set() 

encoder = OneHotEncoder(min_frequency=1, handle_unknown='infrequent_if_exist')
scaler = StandardScaler()
text_pipeline = Pipeline([
    ('vect', TfidfVectorizer(
        max_features=30000,
        stop_words='english',
        ngram_range=(1, 2),
        max_df=0.6,
        min_df=5
    )),
    ('svd', TruncatedSVD(n_components=300, algorithm='arpack', random_state=RANDOM_SEED)) 
])

# 2. Ciclo su ogni classe per estrarre le sue "parole d'oro"
print("Estrazione vocabolario specifico per classe...")
classes = np.unique(y_train)

for label in classes:
    # Prendi solo il testo di QUESTA classe
    subset_text = X_train[y_train == label]['text']
    
    # Fitta un vectorizer piccolo SOLO su questa classe
    temp_vectorizer = TfidfVectorizer(
        max_features=TOP_K_PER_CLASS,
        stop_words='english',
        ngram_range=(1, 2),
        min_df=2 # Qui puoi essere più permissivo perché siamo nel subset
    )
    temp_vectorizer.fit(subset_text)
    
    # Aggiungi le parole trovate al calderone comune
    words = temp_vectorizer.get_feature_names_out()
    final_vocabulary.update(words)
    print(f"Classe {label}: trovate {len(words)} parole specifiche.")

# 3. Converti il set in lista per passarlo a Sklearn
final_vocabulary_list = list(final_vocabulary)
print(f"Vocabolario totale ottimizzato: {len(final_vocabulary_list)} parole uniche.")


master_vectorizer = TfidfVectorizer(
    vocabulary=final_vocabulary_list,  # <--- IL TRUCCO È QUI
    stop_words='english',
    ngram_range=(1, 2)
)

# 5. Ora usa questo nel tuo ColumnTransformer come prima
preprocessor_custom = ColumnTransformer(transformers=[
    ('source', encoder, ['source']),
    ('text', master_vectorizer, 'text'), # Usa il master
], remainder='drop', n_jobs=-1)

Estrazione vocabolario specifico per classe...
Classe 0: trovate 7000 parole specifiche.
Classe 1: trovate 7000 parole specifiche.
Classe 2: trovate 7000 parole specifiche.
Classe 3: trovate 7000 parole specifiche.
Classe 4: trovate 7000 parole specifiche.
Classe 5: trovate 7000 parole specifiche.
Classe 6: trovate 7000 parole specifiche.
Vocabolario totale ottimizzato: 23159 parole uniche.


In [24]:
# 1. Configurazione XGBoost: GLI DIAMO TUTTA LA POTENZA
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

clf_xgb = XGBClassifier(
    tree_method="hist", 
    n_estimators=1000, 
    max_depth=6, 
    learning_rate=0.05,
    n_jobs=-1,          # <--- LUI DEVE USARE TUTTI I CORE (100% CPU)
    random_state=42
)

# 2. Configurazione SVM
clf_svm = SGDClassifier(
    loss='modified_huber', 
    penalty='l2', 
    alpha=1e-4, 
    random_state=42, 
    max_iter=1000, 
    tol=1e-3,
    class_weight='balanced',
    n_jobs=-1           # Anche lui può usare tutto quando tocca a lui
)

# 3. Configurazione Voting: NON PARALLELIZZARE QUI!
voting_clf = VotingClassifier(
    estimators=[
        ('xgb', clf_xgb), 
        ('svm', clf_svm)
    ],
    voting='soft',
    weights=[1, 1],
    n_jobs=1  # <--- FONDAMENTALE: 1 significa "Esegui uno dopo l'altro"
)

# Definiamo cosa vogliamo testare
param_dist = {
    # Parametri per XGBoost
    'xgb__n_estimators': [500, 1000, 1500],
    'xgb__max_depth': [3, 6, 9],
    'xgb__learning_rate': [0.01, 0.05, 0.1],
    'xgb__subsample': [0.7, 0.8, 0.9],
    
    # Parametri per SVM (SGDClassifier)
    'svm__alpha': np.logspace(-5, -2, 4),
    'svm__penalty': ['l2', 'elasticnet'],
    
    # Pesi del Voting
    'weights': [[1, 1], [2, 1], [1, 2]]
}

In [27]:
# Creiamo il tuner
random_search = RandomizedSearchCV(
    estimator=voting_clf,
    param_distributions=param_dist,
    n_iter=20,           # Numero di combinazioni casuali da testare
    cv=3,                # Cross-validation a 3 fold (veloce per iniziare)
    scoring='f1_weighted',
    n_jobs=-1,           # <--- Fondamentale: usa tutti i core del Mac per i diversi test
    verbose=3,
    random_state=42
)

# Avvio del tuning
print("Inizio Hyperparameter Tuning (questo richiederà tempo)...")
X_train_transformed = preprocessor_custom.fit_transform(X_train)
random_search.fit(X_train_transformed, y_train)

# Risultati
print(f"Migliori parametri trovati: {random_search.best_params_}")
print(f"Miglior Score: {random_search.best_score_}")

# Usa il miglior modello trovato
best_voting_clf = random_search.best_estimator_

Inizio Hyperparameter Tuning (questo richiederà tempo)...
Fitting 3 folds for each of 20 candidates, totalling 60 fits


KeyboardInterrupt: 

In [ ]:
print("📊 Valutazione...")
y_pred = voting_clf.predict(X_test_transformed)
print(classification_report(y_test, y_pred))

📊 Valutazione...
              precision    recall  f1-score   support

           0       0.87      0.61      0.72      4438
           1       0.73      0.86      0.79      1848
           2       0.85      0.84      0.84      2119
           3       0.58      0.60      0.59      1778
           4       0.81      0.95      0.88      1663
           5       0.50      0.58      0.54      2117
           6       0.57      0.87      0.69       554

    accuracy                           0.72     14517
   macro avg       0.70      0.76      0.72     14517
weighted avg       0.74      0.72      0.72     14517



In [ ]:
#weights = compute_sample_weight(class_weight='balanced', y=y_train)
## Definisci il modello XGBoost
#clf = XGBClassifier(
#    n_estimators=1000,      # Numero di alberi
#    learning_rate=0.1,     # Impara piano per essere preciso
#    max_depth=6,            # Profondità degli alberi
#    subsample=0.8,          # Usa solo l'80% dei dati per ogni albero (evita overfitting)
#    colsample_bytree=0.8,   # Usa solo l'80% delle feature per albero
#    n_jobs=-1,              # Usa tutti i core
#    random_state=42,
#    tree_method="hist"      # Molto più veloce su dataset grandi
#    # Non serve class_weight='balanced', XGBoost gestisce meglio.
#    # Se vuoi forzarlo, usa scale_pos_weight ma è complesso per multiclasse.
#)
#
#X_train = preprocessor_custom.fit_transform(X_train)
#
## Fit
#clf.fit(X_train, y_train, sample_weight=weights)
#X_test = preprocessor_custom.transform(X_test)
#y_pred = clf.predict(X_test)
#
#
#print(classification_report(y_test, y_pred))

KeyboardInterrupt: 

In [ ]:
#X_train = preprocessor_custom.fit_transform(X_train)
#clf = LogisticRegression(random_state=RANDOM_SEED,C=1, penalty='l2', solver='saga', class_weight='balanced', n_jobs=-1).fit(X_train, y_train)
#X_test = preprocessor_custom.transform(X_test)
#y_pred = clf.predict(X_test)
#
#print(classification_report(y_test, y_pred))

In [ ]:
y_train_pred = clf.predict(X_train)
print(classification_report(y_train, y_train_pred))


              precision    recall  f1-score   support

           0       0.97      0.83      0.90     17752
           1       0.88      0.96      0.92      7394
           2       0.95      0.95      0.95      8478
           3       0.86      0.88      0.87      7110
           4       0.91      0.99      0.95      6652
           5       0.82      0.86      0.84      8465
           6       0.80      1.00      0.89      2217

    accuracy                           0.90     58068
   macro avg       0.88      0.93      0.90     58068
weighted avg       0.91      0.90      0.90     58068

